# Momentum with Holdout Validation

A momentum strategy with a separate strategy validation stage.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
ENTROPY-WEIGHTED MOMENTUM STRATEGY - WALK-FORWARD PRODUCTION VERSION
====================================================================

STRATEGY CONCEPT:
-----------------
Uses entropy of return distribution as proxy for signal reliability:
- Higher entropy = Noisier, chaotic trend → Lower weight (less reliable)
- Lower entropy = Cleaner, stable trend → Higher weight (more reliable)

THEORETICAL BASIS:
------------------
Information Theory: Entropy measures randomness/disorder in data
- High entropy: Returns are uniformly distributed (no clear pattern)
- Low entropy: Returns concentrated around specific values (clear trend)
- Reliability = 1/entropy penalizes unstable trends

EXECUTION LOGIC:
----------------
Day T-1 (Close):
  1. Calculate momentum: Price[T-1]/Price[T-63] - 1
  2. Calculate entropy from 63-day return histogram
  3. Compute reliability = 1/entropy
  4. Generate signal: weight = momentum * reliability
  5. Size positions using Close[T-1] portfolio value

Day T (Open):
  1. Execute signals at Open[T]
  2. Apply transaction costs
  3. Update positions

Day T (Close):
  1. Value portfolio at Close[T]
  2. Calculate returns: Close[T]/Close[T-1] - 1

NO LOOK-AHEAD BIAS GUARANTEE:
------------------------------
- All features use data strictly up to T-1
- Portfolio sizing uses Close[T-1] value
- Execution at Open[T] (realistic 1-day lag)
- No intraday data usage
- No forward/backward filling
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy.stats import entropy as scipy_entropy
from scipy.optimize import minimize
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
import warnings

warnings.filterwarnings("ignore")


class StrategyConfig:
    """
    Centralized configuration for all strategy parameters.

    DESIGN PRINCIPLE: All parameters configurable from one place.
    No hardcoded values scattered throughout code.
    """

    END_DATE = "2018-12-31"
    START_DATE = "2013-01-01"
    INITIAL_CAPITAL = 100000.0
    BENCHMARK_TICKER = "URTH"
    MOMENTUM_LOOKBACK = 63
    ENTROPY_WINDOW = 63
    HISTOGRAM_BINS = 10
    MIN_ENTROPY = 0.1
    MAX_ENTROPY = 3.0
    MIN_ASSETS = 5
    MAX_ASSETS = 15
    MIN_WEIGHT = 0.05
    MAX_WEIGHT = 0.2
    TARGET_EXPOSURE = 1.0
    REBALANCE_FREQUENCY = 21
    MIN_WEIGHT_CHANGE = 0.03
    COMMISSION_PER_TRADE = 0.005
    BID_ASK_SPREAD = 0.0003
    MARKET_IMPACT = 0.0002
    MIN_COMMISSION = 1.0
    MAX_COMMISSION = 5.0
    STOP_LOSS_ENABLED = False
    STOP_LOSS_PCT = -0.15
    ASSET_UNIVERSE = [
        "SPY",
        "QQQ",
        "IWM",
        "DIA",
        "VTI",
        "XLK",
        "XLF",
        "XLV",
        "XLE",
        "XLI",
        "XLY",
        "XLP",
        "XLU",
        "XLRE",
        "XLB",
        "EFA",
        "EEM",
        "VWO",
        "EWJ",
        "EWG",
        "EWU",
        "EWC",
        "AAPL",
        "MSFT",
        "GOOGL",
        "AMZN",
        "META",
        "NVDA",
        "TSLA",
        "TLT",
        "IEF",
        "AGG",
        "LQD",
        "GLD",
        "SLV",
        "USO",
        "VNQ",
        "JNJ",
        "PG",
        "WMT",
        "JPM",
        "BAC",
        "V",
        "MA",
    ]


class DataFetcher:
    """
    Handles all data acquisition from Yahoo Finance.

    CRITICAL PRINCIPLES:
    - No synthetic data generation
    - No forward/backward filling
    - Strict data quality controls
    - Real market data only
    """

    def fetch_historical_data(
        self, ticker: str, start_date: str, end_date: str
    ) -> Optional[pd.DataFrame]:
        """
        Fetch historical OHLCV data from Yahoo Finance.

        CRITICAL: Returns None if insufficient data.
        No fallbacks, no synthetic data.

        Args:
            ticker: Ticker symbol
            start_date: Start date (YYYY-MM-DD)
            end_date: End date (YYYY-MM-DD)

        Returns:
            DataFrame with OHLCV data or None if insufficient
        """
        buffer_start = (pd.to_datetime(start_date) - timedelta(days=150)).strftime("%Y-%m-%d")
        try:
            response_prices = {"historical": download_price_records(ticker, buffer_start, end_date)}
            data = response_prices
            if "historical" not in data or len(data["historical"]) < 100:
                print(f"  ✗ {ticker}: Insufficient data")
                return None
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").reset_index(drop=True)
            df = df.rename(
                columns={
                    "date": "Date",
                    "open": "Open",
                    "high": "High",
                    "low": "Low",
                    "close": "Close",
                    "volume": "Volume",
                }
            )
            required_cols = ["Date", "Open", "Close"]
            for col in required_cols:
                null_pct = df[col].isnull().sum() / len(df)
                if null_pct > 0.05:
                    print(f"  ✗ {ticker}: Excessive missing data ({null_pct:.1%})")
                    return None
            df = df[["Date", "Open", "High", "Low", "Close", "Volume"]]
            return df
        except Exception as e:
            print(f"  ✗ {ticker}: Error - {str(e)}")
            return None

    def fetch_universe_data(
        self, tickers: List[str], start_date: str, end_date: str
    ) -> Dict[str, pd.DataFrame]:
        """
        Fetch data for entire universe.

        Args:
            tickers: List of ticker symbols
            start_date: Start date
            end_date: End date

        Returns:
            Dictionary mapping tickers to DataFrames
        """
        print(f"\nFetching data for {len(tickers)} assets...")
        print(f"Period: {start_date} to {end_date}")
        data_dict = {}
        successful = 0
        for ticker in tickers:
            df = self.fetch_historical_data(ticker, start_date, end_date)
            if df is not None:
                data_dict[ticker] = df
                successful += 1
                if successful <= 5:
                    print(f"  ✓ {ticker}: {len(df)} days")
        print(f"\n✓ Successfully fetched {successful}/{len(tickers)} assets")
        if successful < 5:
            raise ValueError(f"Insufficient assets: {successful} < 5 minimum")
        return data_dict


class EntropyMomentumCalculator:
    """
    Calculate entropy-weighted momentum signals.

    METHODOLOGY:
    ------------
    1. Momentum: Price[T-1]/Price[T-lookback] - 1
    2. Entropy: Information entropy of return distribution
    3. Reliability: 1/entropy (inverse relationship)
    4. Weight: momentum * reliability

    ENTROPY INTERPRETATION:
    ----------------------
    - Low entropy (e.g., 0.5): Returns concentrated, clear trend
      → High reliability → Larger weight
    - High entropy (e.g., 2.0): Returns dispersed, noisy
      → Low reliability → Smaller weight

    ANTI LOOK-AHEAD:
    ----------------
    All calculations use data strictly up to T-1 for T signals.
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.momentum_lookback = config.MOMENTUM_LOOKBACK
        self.entropy_window = config.ENTROPY_WINDOW
        self.histogram_bins = config.HISTOGRAM_BINS
        self.min_entropy = config.MIN_ENTROPY
        self.max_entropy = config.MAX_ENTROPY

    def calculate_entropy_from_returns(self, returns: pd.Series) -> float:
        """
        Calculate Shannon entropy from return distribution.

        FORMULA: H = -Σ(p_i * log(p_i))

        Where p_i is the probability (frequency) of returns in bin i.

        INTERPRETATION:
        - Uniform distribution → Maximum entropy (high uncertainty)
        - Concentrated distribution → Low entropy (high certainty)

        Args:
            returns: Series of returns for entropy calculation

        Returns:
            Entropy value (float)
        """
        clean_returns = returns.dropna()
        if len(clean_returns) < 10:
            return self.max_entropy
        counts, _ = np.histogram(clean_returns, bins=self.histogram_bins)
        probabilities = counts / counts.sum()
        probabilities = probabilities[probabilities > 0]
        entropy_val = scipy_entropy(probabilities)
        entropy_val = np.clip(entropy_val, self.min_entropy, self.max_entropy)
        return entropy_val

    def calculate_features_t_minus_1(self, price_data: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate momentum and entropy features using STRICT T-1 methodology.

        CRITICAL ANTI LOOK-AHEAD DESIGN:
        ---------------------------------
        At time T, we can only know:
        - Prices up to Close[T-1]
        - Returns up to T-1

        Therefore:
        1. Calculate returns: Close[t]/Close[t-1] - 1
        2. Calculate momentum at T-1: Price[T-1]/Price[T-1-lookback]
        3. Calculate entropy at T-1: Using returns [T-1-window : T-1]
        4. LAG features by 1 day: Ensures T-1 availability for T signals

        Args:
            price_data: DataFrame with Date and Close columns

        Returns:
            DataFrame with lagged momentum and entropy features
        """
        df = price_data.copy()
        df = df.sort_values("Date").reset_index(drop=True)
        df["Returns"] = df["Close"].pct_change()
        df["Momentum_Raw"] = df["Close"] / df["Close"].shift(self.momentum_lookback) - 1
        entropy_values = []
        for i in range(len(df)):
            if i < self.entropy_window:
                entropy_values.append(np.nan)
            else:
                hist_returns = df["Returns"].iloc[i - self.entropy_window : i]
                entropy_val = self.calculate_entropy_from_returns(hist_returns)
                entropy_values.append(entropy_val)
        df["Entropy_Raw"] = entropy_values
        df["Reliability_Raw"] = 1.0 / df["Entropy_Raw"]
        df["Signal_Raw"] = df["Momentum_Raw"] * df["Reliability_Raw"]
        df["Momentum"] = df["Momentum_Raw"].shift(1)
        df["Entropy"] = df["Entropy_Raw"].shift(1)
        df["Reliability"] = df["Reliability_Raw"].shift(1)
        df["Signal"] = df["Signal_Raw"].shift(1)
        df = df.drop(["Momentum_Raw", "Entropy_Raw", "Reliability_Raw", "Signal_Raw"], axis=1)
        return df

    def calculate_universe_signals(self, data_dict: Dict[str, pd.DataFrame]) -> pd.DataFrame:
        """
        Calculate entropy-momentum signals for entire universe.

        Process:
        1. Calculate features for each asset
        2. Align all assets on common dates
        3. Create signal matrix

        Args:
            data_dict: Dictionary mapping tickers to price DataFrames

        Returns:
            DataFrame with signals for all assets (dates x tickers)
        """
        print("\nCalculating entropy-weighted momentum signals...")
        signal_dict = {}
        momentum_dict = {}
        entropy_dict = {}
        for ticker, price_df in data_dict.items():
            features_df = self.calculate_features_t_minus_1(price_df)
            features_df = features_df.set_index("Date")
            signal_dict[ticker] = features_df["Signal"]
            momentum_dict[ticker] = features_df["Momentum"]
            entropy_dict[ticker] = features_df["Entropy"]
        signals_df = pd.DataFrame(signal_dict)
        momentum_df = pd.DataFrame(momentum_dict)
        entropy_df = pd.DataFrame(entropy_dict)
        print(f"✓ Signals calculated for {len(signals_df.columns)} assets")
        print(f"  Date range: {signals_df.index.min().date()} to {signals_df.index.max().date()}")
        print(f"  Total days: {len(signals_df)}")
        signals_df.momentum = momentum_df
        signals_df.entropy = entropy_df
        return signals_df


class EntropyMomentumOptimizer:
    """
    Portfolio optimizer for entropy-weighted momentum strategy.

    ALLOCATION METHOD:
    -----------------
    1. Rank assets by entropy-weighted signal
    2. Select top N assets (configurable)
    3. Weight proportionally to signal strength
    4. Apply position limits (min/max weights)
    5. Normalize to target exposure

    RISK CONTROLS:
    --------------
    - Position limits prevent concentration
    - Minimum positions for diversification
    - Only positive signals eligible (no shorts)
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.min_assets = config.MIN_ASSETS
        self.max_assets = config.MAX_ASSETS
        self.min_weight = config.MIN_WEIGHT
        self.max_weight = config.MAX_WEIGHT
        self.target_exposure = config.TARGET_EXPOSURE

    def optimize_portfolio(
        self, signals: pd.Series, current_positions: Optional[Dict] = None
    ) -> Dict[str, float]:
        """
        Optimize portfolio weights based on entropy-momentum signals.

        ALLOCATION LOGIC:
        ----------------
        1. Filter: Only positive signals (bullish momentum + high reliability)
        2. Rank: Sort by signal strength
        3. Select: Top N assets (between min_assets and max_assets)
        4. Weight: Proportional to signal strength
        5. Constrain: Apply min/max position limits
        6. Normalize: Scale to target exposure

        Args:
            signals: Current signals for all assets
            current_positions: Current portfolio weights (for stability)

        Returns:
            Dictionary mapping tickers to target weights
        """
        positive_signals = signals[signals > 0].dropna()
        if len(positive_signals) == 0:
            return {}
        ranked_signals = positive_signals.sort_values(ascending=False)
        n_select = min(self.max_assets, len(ranked_signals))
        n_select = (
            max(n_select, self.min_assets)
            if len(ranked_signals) >= self.min_assets
            else len(ranked_signals)
        )
        top_signals = ranked_signals.head(n_select)
        raw_weights = top_signals / top_signals.sum()
        constrained_weights = raw_weights.clip(lower=self.min_weight, upper=self.max_weight)
        total_weight = constrained_weights.sum()
        if total_weight > 0:
            final_weights = constrained_weights / total_weight * self.target_exposure
        else:
            return {}
        if current_positions is not None:
            significant_change = False
            for ticker in set(list(final_weights.index) + list(current_positions.keys())):
                old_weight = current_positions.get(ticker, 0)
                new_weight = final_weights.get(ticker, 0)
                if abs(new_weight - old_weight) > self.config.MIN_WEIGHT_CHANGE:
                    significant_change = True
                    break
            if not significant_change:
                return current_positions
        return final_weights.to_dict()


class BacktestEngine:
    """
    Realistic backtesting engine with proper execution flow.

    EXECUTION FLOW:
    ---------------
    Day T-1 (Close):
      - Portfolio valued at Close[T-1]
      - Features calculated from data up to T-1
      - Signals generated
      - Target weights determined using Close[T-1] portfolio value

    Day T (Open):
      - Execute trades at Open[T]
      - Pay transaction costs
      - Update positions

    Day T (Close):
      - Portfolio valued at Close[T]
      - Returns: Close[T]/Close[T-1] - 1

    TRANSACTION COSTS:
    ------------------
    - Commission per trade
    - Bid-ask spread
    - Market impact
    - All costs applied at execution (Open[T])
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.trades = []

    def calculate_transaction_costs(self, trade_value: float) -> float:
        """
        Calculate realistic transaction costs.

        Components:
        - Commission: percentage of trade value (capped)
        - Spread: cost of crossing bid-ask spread
        - Impact: price movement from order

        Args:
            trade_value: Dollar value of trade

        Returns:
            Total transaction cost
        """
        abs_value = abs(trade_value)
        commission = abs_value * self.config.COMMISSION_PER_TRADE
        commission = np.clip(commission, self.config.MIN_COMMISSION, self.config.MAX_COMMISSION)
        spread_cost = abs_value * self.config.BID_ASK_SPREAD
        impact_cost = abs_value * self.config.MARKET_IMPACT
        return commission + spread_cost + impact_cost

    def run_backtest(
        self,
        data_dict: Dict[str, pd.DataFrame],
        signals_df: pd.DataFrame,
        optimizer: EntropyMomentumOptimizer,
    ) -> Dict:
        """
        Execute backtest with realistic trading.

        Args:
            data_dict: Price data for all assets
            signals_df: Entropy-momentum signals
            optimizer: Portfolio optimizer

        Returns:
            Comprehensive results dictionary
        """
        print("\n" + "=" * 80)
        print("EXECUTING BACKTEST")
        print("=" * 80)
        backtest_dates = signals_df.index
        start_date = pd.to_datetime(self.config.START_DATE)
        end_date = pd.to_datetime(self.config.END_DATE)
        backtest_dates = backtest_dates[
            (backtest_dates >= start_date) & (backtest_dates <= end_date)
        ]
        if len(backtest_dates) < 50:
            print(f"ERROR: Insufficient backtest dates ({len(backtest_dates)})")
            return {
                "portfolio_history": pd.DataFrame(),
                "trade_history": [],
                "metrics": {
                    "initial_capital": self.config.INITIAL_CAPITAL,
                    "final_value": self.config.INITIAL_CAPITAL,
                    "total_return": 0.0,
                    "annual_return": 0.0,
                    "volatility": 0.0,
                    "sharpe_ratio": 0.0,
                    "max_drawdown": 0.0,
                    "total_costs": 0.0,
                    "num_trades": 0,
                },
                "signals_df": signals_df,
                "data_dict": data_dict,
            }
        print(f"Backtest period: {backtest_dates[0].date()} to {backtest_dates[-1].date()}")
        print(f"Trading days: {len(backtest_dates)}")
        capital = self.config.INITIAL_CAPITAL
        positions = {}
        portfolio_history = []
        last_rebalance = -self.config.REBALANCE_FREQUENCY
        total_costs = 0.0
        for day_idx, signal_date in enumerate(backtest_dates[:-1]):
            current_prices = {}
            for ticker in signals_df.columns:
                if ticker in data_dict:
                    ticker_df = data_dict[ticker]
                    matches = ticker_df[ticker_df["Date"] == signal_date]
                    if not matches.empty:
                        current_prices[ticker] = matches.iloc[0]["Close"]
            risky_value = sum(
                (positions.get(ticker, 0) * current_prices.get(ticker, 0) for ticker in positions)
            )
            portfolio_value = risky_value
            if day_idx == 0:
                portfolio_value = capital
            if day_idx % 50 == 0:
                print(
                    f"  Day {day_idx + 1}/{len(backtest_dates) - 1}: {signal_date.date()} | Value: ${portfolio_value:,.0f}"
                )
            days_since_rebal = day_idx - last_rebalance
            should_rebalance = days_since_rebal >= self.config.REBALANCE_FREQUENCY
            if should_rebalance:
                current_signals = signals_df.loc[signal_date]
                current_weights = {}
                for ticker, shares in positions.items():
                    if ticker in current_prices and portfolio_value > 0:
                        current_weights[ticker] = shares * current_prices[ticker] / portfolio_value
                target_weights = optimizer.optimize_portfolio(current_signals, current_weights)
                exec_idx = day_idx + 1
                if exec_idx >= len(backtest_dates):
                    continue
                exec_date = backtest_dates[exec_idx]
                exec_prices = {}
                for ticker in signals_df.columns:
                    if ticker in data_dict:
                        ticker_df = data_dict[ticker]
                        matches = ticker_df[ticker_df["Date"] == exec_date]
                        if not matches.empty:
                            exec_prices[ticker] = matches.iloc[0]["Open"]
                day_costs = 0.0
                for ticker in list(positions.keys()):
                    if ticker not in target_weights and ticker in exec_prices:
                        shares = positions[ticker]
                        value = shares * exec_prices[ticker]
                        cost = self.calculate_transaction_costs(value)
                        capital += value - cost
                        day_costs += cost
                        del positions[ticker]
                        self.trades.append(
                            {
                                "date": exec_date,
                                "ticker": ticker,
                                "action": "SELL",
                                "shares": shares,
                                "price": exec_prices[ticker],
                                "value": value,
                                "cost": cost,
                            }
                        )
                for ticker, target_weight in target_weights.items():
                    if ticker not in exec_prices:
                        continue
                    target_value = portfolio_value * target_weight
                    current_value = positions.get(ticker, 0) * exec_prices[ticker]
                    trade_value = target_value - current_value
                    if abs(trade_value) > portfolio_value * 0.01:
                        shares_to_trade = trade_value / exec_prices[ticker]
                        cost = self.calculate_transaction_costs(abs(trade_value))
                        positions[ticker] = positions.get(ticker, 0) + shares_to_trade
                        capital -= trade_value + cost
                        day_costs += cost
                        self.trades.append(
                            {
                                "date": exec_date,
                                "ticker": ticker,
                                "action": "BUY" if shares_to_trade > 0 else "SELL",
                                "shares": abs(shares_to_trade),
                                "price": exec_prices[ticker],
                                "value": abs(trade_value),
                                "cost": cost,
                            }
                        )
                total_costs += day_costs
                last_rebalance = day_idx
            portfolio_history.append(
                {
                    "Date": signal_date,
                    "Value": portfolio_value,
                    "Positions": len(positions),
                    "Cash": capital,
                }
            )
        portfolio_df = pd.DataFrame(portfolio_history)
        portfolio_df["Returns"] = portfolio_df["Value"].pct_change()
        final_value = portfolio_df["Value"].iloc[-1]
        total_return = final_value / self.config.INITIAL_CAPITAL - 1
        days = len(portfolio_df)
        annual_return = (1 + total_return) ** (252 / days) - 1 if days > 0 else 0
        returns = portfolio_df["Returns"].dropna()
        volatility = returns.std() * np.sqrt(252) if len(returns) > 0 else 0
        if volatility > 0.001:
            sharpe = (annual_return - 0.02) / volatility
        else:
            sharpe = 0.0
        cumulative = (1 + returns).cumprod() if len(returns) > 0 else pd.Series([1])
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_dd = drawdown.min() if len(drawdown) > 0 else 0
        results = {
            "portfolio_history": portfolio_df,
            "trade_history": self.trades,
            "metrics": {
                "initial_capital": self.config.INITIAL_CAPITAL,
                "final_value": final_value,
                "total_return": total_return,
                "annual_return": annual_return,
                "volatility": volatility,
                "sharpe_ratio": sharpe,
                "max_drawdown": max_dd,
                "total_costs": total_costs,
                "num_trades": len(self.trades),
            },
            "signals_df": signals_df,
            "data_dict": data_dict,
        }
        print("\n" + "=" * 80)
        print("BACKTEST COMPLETE")
        print("=" * 80)
        print(f"Final Value: ${final_value:,.0f}")
        print(f"Total Return: {total_return:.2%}")
        print(f"Annual Return: {annual_return:.2%}")
        print(f"Sharpe Ratio: {sharpe:.2f}")
        print(f"Max Drawdown: {max_dd:.2%}")
        print(f"Total Costs: ${total_costs:,.0f}")
        print(f"Number of Trades: {len(self.trades)}")
        print("=" * 80)
        return results


def create_visualizations(results: Dict, config: StrategyConfig):
    """
    Create comprehensive visualization dashboard.

    Panels:
    1. Equity curve vs benchmark
    2. Drawdown profile
    3. Signal strength distribution
    4. Entropy vs momentum scatter
    5. Monthly returns heatmap
    """
    portfolio_df = results["portfolio_history"]
    signals_df = results["signals_df"]
    data_dict = results["data_dict"]
    benchmark_data = None
    if config.BENCHMARK_TICKER in data_dict:
        bench_df = data_dict[config.BENCHMARK_TICKER].copy()
        bench_df = bench_df[bench_df["Date"].isin(portfolio_df["Date"])]
        first_close = bench_df["Close"].iloc[0]
        bench_df["Value"] = bench_df["Close"] / first_close * config.INITIAL_CAPITAL
        benchmark_data = bench_df.set_index("Date")["Value"]
    fig = plt.figure(figsize=(20, 14))
    gs = fig.add_gridspec(4, 2, hspace=0.3, wspace=0.25)
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(
        portfolio_df["Date"],
        portfolio_df["Value"],
        label="Entropy-Momentum Strategy",
        linewidth=2.5,
        color="#2E86AB",
    )
    if benchmark_data is not None:
        ax1.plot(
            benchmark_data.index,
            benchmark_data.values,
            label=f"{config.BENCHMARK_TICKER} (Benchmark)",
            linewidth=2,
            color="#A23B72",
            alpha=0.7,
        )
    ax1.set_title(
        "Portfolio Performance: Entropy-Weighted Momentum", fontsize=16, fontweight="bold"
    )
    ax1.set_ylabel("Portfolio Value ($)", fontsize=12)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))
    ax2 = fig.add_subplot(gs[1, 0])
    returns = portfolio_df["Returns"].dropna()
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    ax2.fill_between(range(len(drawdown)), drawdown * 100, 0, alpha=0.7, color="red")
    ax2.set_title("Drawdown Profile", fontsize=14, fontweight="bold")
    ax2.set_ylabel("Drawdown (%)", fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.plot(portfolio_df["Date"], portfolio_df["Positions"], linewidth=2, color="blue")
    ax3.set_title("Number of Positions Over Time", fontsize=14, fontweight="bold")
    ax3.set_ylabel("Number of Positions", fontsize=11)
    ax3.grid(True, alpha=0.3)
    ax4 = fig.add_subplot(gs[2, 0])
    all_signals = signals_df.values.flatten()
    all_signals = all_signals[~np.isnan(all_signals)]
    ax4.hist(all_signals, bins=50, alpha=0.7, color="purple", edgecolor="black")
    ax4.axvline(x=0, color="red", linestyle="--", linewidth=2)
    ax4.set_title("Signal Strength Distribution", fontsize=14, fontweight="bold")
    ax4.set_xlabel("Entropy-Weighted Signal", fontsize=11)
    ax4.set_ylabel("Frequency", fontsize=11)
    ax4.grid(True, alpha=0.3)
    ax5 = fig.add_subplot(gs[2, 1])
    momentum_df = signals_df.momentum
    entropy_df = signals_df.entropy
    sample_dates = signals_df.index[::20]
    for ticker in signals_df.columns[:10]:
        mom_vals = momentum_df.loc[sample_dates, ticker].dropna()
        ent_vals = entropy_df.loc[sample_dates, ticker].dropna()
        common_idx = mom_vals.index.intersection(ent_vals.index)
        if len(common_idx) > 0:
            ax5.scatter(ent_vals[common_idx], mom_vals[common_idx], alpha=0.3, s=20)
    ax5.set_title("Entropy vs Momentum Relationship", fontsize=14, fontweight="bold")
    ax5.set_xlabel("Entropy (lower = more reliable)", fontsize=11)
    ax5.set_ylabel("Momentum", fontsize=11)
    ax5.axhline(y=0, color="black", linestyle="-", linewidth=0.5)
    ax5.grid(True, alpha=0.3)
    ax6 = fig.add_subplot(gs[3, :])
    portfolio_df["Year"] = pd.to_datetime(portfolio_df["Date"]).dt.year
    portfolio_df["Month"] = pd.to_datetime(portfolio_df["Date"]).dt.month
    monthly_rets = (
        portfolio_df.groupby(["Year", "Month"])
        .apply(lambda x: (1 + x["Returns"]).prod() - 1)
        .reset_index(name="Return")
    )
    heatmap_data = monthly_rets.pivot(index="Year", columns="Month", values="Return")
    month_names = [
        "Jan",
        "Feb",
        "Mar",
        "Apr",
        "May",
        "Jun",
        "Jul",
        "Aug",
        "Sep",
        "Oct",
        "Nov",
        "Dec",
    ]
    heatmap_data.columns = [month_names[int(m) - 1] for m in heatmap_data.columns]
    sns.heatmap(
        heatmap_data * 100,
        annot=True,
        fmt=".1f",
        cmap="RdYlGn",
        center=0,
        ax=ax6,
        cbar_kws={"label": "Monthly Return (%)"},
    )
    ax6.set_title("Monthly Returns Heatmap", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


class StrategyValidator:
    """
    Comprehensive validation framework for entropy-momentum strategy.

    VALIDATION CHECKS:
    ------------------
    1. Pricing methodology robustness (adjusted vs unadjusted)
    2. Cash-inclusive valuation verification
    3. Entropy calculation method sensitivity
    4. Out-of-sample walk-forward stability
    5. Asset/sector contribution analysis
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.validation_results = {}

    def test_entropy_methods(self, returns: pd.Series) -> Dict:
        """
        Test different entropy calculation methods.

        Methods tested:
        1. Fixed-width histogram (current implementation)
        2. Fixed-edge histogram (predetermined ranges)
        3. KDE-based entropy (kernel density estimation)

        Args:
            returns: Return series for entropy calculation

        Returns:
            Dictionary with entropy values from each method
        """
        clean_returns = returns.dropna()
        if len(clean_returns) < 10:
            return {"fixed_width": np.nan, "fixed_edge": np.nan, "kde": np.nan}
        counts_fw, _ = np.histogram(clean_returns, bins=self.config.HISTOGRAM_BINS)
        probs_fw = counts_fw[counts_fw > 0] / counts_fw.sum()
        entropy_fw = scipy_entropy(probs_fw)
        std = clean_returns.std()
        edges = np.array([-3, -2, -1, 0, 1, 2, 3]) * std
        counts_fe, _ = np.histogram(clean_returns, bins=edges)
        probs_fe = counts_fe[counts_fe > 0] / counts_fe.sum()
        entropy_fe = scipy_entropy(probs_fe)
        from scipy.stats import gaussian_kde

        try:
            kde = gaussian_kde(clean_returns)
            x_eval = np.linspace(clean_returns.min(), clean_returns.max(), 100)
            density = kde(x_eval)
            dx = x_eval[1] - x_eval[0]
            probs_kde = density * dx
            probs_kde = probs_kde[probs_kde > 0]
            entropy_kde = scipy_entropy(probs_kde)
        except:
            entropy_kde = np.nan
        return {"fixed_width": entropy_fw, "fixed_edge": entropy_fe, "kde": entropy_kde}

    def calculate_turnover(self, trade_history: List[Dict], portfolio_values: pd.Series) -> float:
        """
        Calculate portfolio turnover rate.

        Turnover = Sum(|Trade Value|) / Average Portfolio Value

        High turnover indicates frequent trading (higher costs).
        """
        if not trade_history:
            return 0.0
        total_trade_value = sum((abs(trade["value"]) for trade in trade_history))
        avg_portfolio_value = portfolio_values.mean()
        if avg_portfolio_value > 0:
            return total_trade_value / avg_portfolio_value
        return 0.0

    def calculate_hit_rate(
        self, trade_history: List[Dict], data_dict: Dict[str, pd.DataFrame]
    ) -> Dict:
        """
        Calculate hit rate (% of winning trades).

        Returns detailed breakdown by action type and asset.
        """
        if not trade_history:
            return {"overall": 0.0}
        sell_trades = [t for t in trade_history if t["action"] == "SELL"]
        if not sell_trades:
            return {"overall": 0.0}
        wins = 0
        total = 0
        for trade in sell_trades:
            ticker = trade["ticker"]
            exit_date = trade["date"]
            exit_price = trade["price"]
            buy_trades = [
                t
                for t in trade_history
                if t["ticker"] == ticker and t["action"] == "BUY" and (t["date"] < exit_date)
            ]
            if buy_trades:
                buy_trade = buy_trades[-1]
                entry_price = buy_trade["price"]
                ret = (exit_price - entry_price) / entry_price
                if ret > 0:
                    wins += 1
                total += 1
        hit_rate = wins / total if total > 0 else 0.0
        return {"overall": hit_rate, "total_trades": total, "winning_trades": wins}

    def run_walk_forward_validation(
        self,
        data_dict: Dict[str, pd.DataFrame],
        calculator: EntropyMomentumCalculator,
        optimizer: EntropyMomentumOptimizer,
        engine: BacktestEngine,
    ) -> Dict:
        """
        Out-of-sample walk-forward validation across calendar years.

        Process:
        1. Split data by calendar years
        2. Run strategy on each year independently
        3. Compare performance metrics across years
        4. Check for consistency in hit rates and Sharpe

        No parameter changes between periods.
        """
        print("\n" + "=" * 80)
        print("WALK-FORWARD VALIDATION (OUT-OF-SAMPLE)")
        print("=" * 80)
        all_dates = []
        for df in data_dict.values():
            all_dates.extend(df["Date"].tolist())
        all_dates = sorted(set(all_dates))
        min_date = min(all_dates)
        max_date = max(all_dates)
        years = list(range(min_date.year, max_date.year + 1))
        yearly_results = {}
        for year in years:
            year_start = pd.Timestamp(f"{year}-01-01")
            year_end = pd.Timestamp(f"{year}-12-31")
            buffer_start = year_start - timedelta(days=150)
            year_data = {}
            for ticker, df in data_dict.items():
                year_df = df[(df["Date"] >= buffer_start) & (df["Date"] <= year_end)].copy()
                if len(year_df) >= 100:
                    year_data[ticker] = year_df
            if len(year_data) < 5:
                print(f"  Year {year}: Insufficient data ({len(year_data)} assets)")
                continue
            print(f"\n  Processing {year}: {len(year_data)} assets")
            signals_df = calculator.calculate_universe_signals(year_data)
            signals_year = signals_df[signals_df.index >= year_start]
            if len(signals_year) < 50:
                print(f"    Insufficient trading days ({len(signals_year)})")
                continue
            year_engine = BacktestEngine(self.config)
            year_results = year_engine.run_backtest(year_data, signals_year, optimizer)
            portfolio_hist = year_results.get("portfolio_history", pd.DataFrame())
            if portfolio_hist.empty or "Value" not in portfolio_hist.columns:
                print(f"    Year {year}: Backtest failed (empty results)")
                continue
            metrics = year_results["metrics"]
            turnover = self.calculate_turnover(
                year_results["trade_history"], portfolio_hist["Value"]
            )
            hit_rate_data = self.calculate_hit_rate(year_results["trade_history"], year_data)
            yearly_results[year] = {
                "annual_return": metrics["annual_return"],
                "volatility": metrics["volatility"],
                "sharpe": metrics["sharpe_ratio"],
                "max_drawdown": metrics["max_drawdown"],
                "turnover": turnover,
                "hit_rate": hit_rate_data["overall"],
                "num_trades": metrics["num_trades"],
            }
            print(
                f"    Return: {metrics['annual_return']:>7.2%} | Sharpe: {metrics['sharpe_ratio']:>5.2f} | Turnover: {turnover:>5.2f}x | Hit Rate: {hit_rate_data['overall']:>5.1%}"
            )
        return yearly_results

    def validate_entropy_robustness(
        self, data_dict: Dict[str, pd.DataFrame], config: StrategyConfig
    ) -> pd.DataFrame:
        """
        Test sensitivity to entropy calculation methods.

        Compares:
        - Fixed-width histogram
        - Fixed-edge histogram
        - KDE-based entropy

        Checks correlation between methods to ensure consistency.
        """
        print("\n" + "=" * 80)
        print("ENTROPY METHOD ROBUSTNESS TEST")
        print("=" * 80)
        results = []
        test_assets = list(data_dict.keys())[:10]
        for ticker in test_assets:
            df = data_dict[ticker]
            df = df.sort_values("Date")
            df["Returns"] = df["Close"].pct_change()
            for i in range(config.ENTROPY_WINDOW, len(df), 63):
                window_returns = df["Returns"].iloc[i - config.ENTROPY_WINDOW : i]
                methods = self.test_entropy_methods(window_returns)
                results.append(
                    {
                        "ticker": ticker,
                        "date": df["Date"].iloc[i],
                        "fixed_width": methods["fixed_width"],
                        "fixed_edge": methods["fixed_edge"],
                        "kde": methods["kde"],
                    }
                )
        results_df = pd.DataFrame(results)
        print("\nEntropy Method Correlations:")
        corr_matrix = results_df[["fixed_width", "fixed_edge", "kde"]].corr()
        print(corr_matrix)
        if corr_matrix.loc["fixed_width", "fixed_edge"] > 0.7:
            print("\n✓ High correlation between fixed-width and fixed-edge (stable)")
        else:
            print("\n⚠ Low correlation between methods (potential sensitivity)")
        return results_df

    def run_full_validation(
        self,
        data_dict: Dict[str, pd.DataFrame],
        calculator: EntropyMomentumCalculator,
        optimizer: EntropyMomentumOptimizer,
        baseline_results: Dict,
    ) -> Dict:
        """
        Execute complete validation suite.

        Returns comprehensive validation report.
        """
        print("\n" + "=" * 80)
        print("COMPREHENSIVE VALIDATION SUITE")
        print("=" * 80)
        validation_report = {}
        baseline_metrics = baseline_results["metrics"]
        validation_report["baseline"] = {
            "sharpe": baseline_metrics["sharpe_ratio"],
            "max_drawdown": baseline_metrics["max_drawdown"],
            "turnover": self.calculate_turnover(
                baseline_results["trade_history"], baseline_results["portfolio_history"]["Value"]
            ),
        }
        print(f"\nBaseline Performance:")
        print(f"  Sharpe Ratio: {baseline_metrics['sharpe_ratio']:.3f}")
        print(f"  Max Drawdown: {baseline_metrics['max_drawdown']:.2%}")
        print(f"  Turnover: {validation_report['baseline']['turnover']:.2f}x")
        entropy_results = self.validate_entropy_robustness(data_dict, self.config)
        validation_report["entropy_robustness"] = entropy_results
        engine = BacktestEngine(self.config)
        yearly_results = self.run_walk_forward_validation(data_dict, calculator, optimizer, engine)
        validation_report["yearly_performance"] = yearly_results
        if yearly_results:
            sharpes = [y["sharpe"] for y in yearly_results.values() if not np.isnan(y["sharpe"])]
            returns = [y["annual_return"] for y in yearly_results.values()]
            turnovers = [y["turnover"] for y in yearly_results.values()]
            print("\n" + "=" * 80)
            print("STABILITY ANALYSIS")
            print("=" * 80)
            print(f"\nSharpe Ratio:")
            print(f"  Mean: {np.mean(sharpes):.3f}")
            print(f"  Std Dev: {np.std(sharpes):.3f}")
            print(f"  Min/Max: {np.min(sharpes):.3f} / {np.max(sharpes):.3f}")
            print(f"\nAnnual Returns:")
            print(f"  Mean: {np.mean(returns):.2%}")
            print(f"  Std Dev: {np.std(returns):.2%}")
            print(f"\nTurnover:")
            print(f"  Mean: {np.mean(turnovers):.2f}x")
            print(f"  Std Dev: {np.std(turnovers):.2f}x")
            sharpe_stable = (
                np.std(sharpes) / abs(np.mean(sharpes)) < 0.5 if np.mean(sharpes) != 0 else False
            )
            turnover_stable = (
                np.std(turnovers) / np.mean(turnovers) < 0.3 if np.mean(turnovers) > 0 else False
            )
            validation_report["stability"] = {
                "sharpe_cv": (
                    np.std(sharpes) / abs(np.mean(sharpes)) if np.mean(sharpes) != 0 else np.inf
                ),
                "turnover_cv": (
                    np.std(turnovers) / np.mean(turnovers) if np.mean(turnovers) > 0 else np.inf
                ),
                "sharpe_stable": sharpe_stable,
                "turnover_stable": turnover_stable,
            }
            print(f"\n{'=' * 80}")
            print("VALIDATION SUMMARY")
            print(f"{'=' * 80}")
            print(f"Sharpe Stability: {('✓ PASS' if sharpe_stable else '✗ FAIL')}")
            print(f"Turnover Stability: {('✓ PASS' if turnover_stable else '✗ FAIL')}")
        return validation_report


def main():
    """Execute entropy-weighted momentum strategy with full validation."""
    print("=" * 80)
    print("ENTROPY-WEIGHTED MOMENTUM STRATEGY")
    print("=" * 80)
    print("\nConcept: Information Theory Applied to Momentum Trading")
    print("Entropy = Measure of uncertainty in return distribution")
    print("Strategy: Weight momentum by inverse entropy (reliability)")
    print("=" * 80)
    config = StrategyConfig()
    print(f"\nConfiguration:")
    print(f"  Period: {config.START_DATE} to {config.END_DATE}")
    print(f"  Capital: ${config.INITIAL_CAPITAL:,.0f}")
    print(f"  Momentum Lookback: {config.MOMENTUM_LOOKBACK} days")
    print(f"  Entropy Window: {config.ENTROPY_WINDOW} days")
    print(f"  Rebalance: Every {config.REBALANCE_FREQUENCY} days")
    print(f"  Universe Size: {len(config.ASSET_UNIVERSE)} assets")
    print("\n" + "=" * 80)
    print("STEP 1: DATA ACQUISITION")
    print("=" * 80)
    fetcher = DataFetcher()
    data_dict = fetcher.fetch_universe_data(
        config.ASSET_UNIVERSE + [config.BENCHMARK_TICKER], config.START_DATE, config.END_DATE
    )
    print("\n" + "=" * 80)
    print("STEP 2: ENTROPY-MOMENTUM SIGNAL GENERATION")
    print("=" * 80)
    calculator = EntropyMomentumCalculator(config)
    signals_df = calculator.calculate_universe_signals(data_dict)
    print("\n" + "=" * 80)
    print("STEP 3: BACKTEST EXECUTION")
    print("=" * 80)
    optimizer = EntropyMomentumOptimizer(config)
    engine = BacktestEngine(config)
    results = engine.run_backtest(data_dict, signals_df, optimizer)
    print("\n" + "=" * 80)
    print("STEP 4: VALIDATION SUITE")
    print("=" * 80)
    validator = StrategyValidator(config)
    validation_report = validator.run_full_validation(data_dict, calculator, optimizer, results)
    print("\n" + "=" * 80)
    print("STEP 5: VISUALIZATION")
    print("=" * 80)
    create_visualizations(results, config)
    return (results, validation_report)


if __name__ == "__main__":
    results, validation = main()
